# UNIVERSAL ADAPTIVE PIPELINE: MULTI-CLASS TEXT CLASSIFICATION
### COMPETITION-GRADE ARCHITECTURE FOR 10 TO 50+ CLASSES WITH EXTREME IMBALANCE

**Prinsip Desain:**
1. **Auto-Detection & Field Fusion**: Otomatis mendeteksi kolom ID, Target, dan satu atau lebih kolom Teks.
2. **Rare-Class Shielding**: Menjamin fungsi pembagian data (`StratifiedKFold` / `train_test_split`) tidak akan melempar *ValueError* jika terdapat kelas minoritas ekstrem yang hanya memiliki 1 sampel.
3. **Adaptive Balanced Class Weights**: Menghitung pembobotan kelas terbalik matematis secara dinamis dengan proteksi clipping agar kelas langka tidak merusak gradien.
4. **Dual Engine**: Baseline Cepat (TF-IDF + Balanced Softmax Logistic Regression) dan Deep Learning (IndoBERT / RoBERTa dengan Custom WeightedTrainer).
5. **Evaluasi Menyeluruh**: Laporan presisi, recall, dan Macro-F1 untuk setiap kategori target.


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

SEED = 2026
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Environment siap. Device: {device} | {torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'}")


## 1. Unified Configuration Card


In [ ]:
CONFIG = {
    # Jalur dataset kompetisi
    "train_path": "cases/nlp_kompas/train.csv",
    "test_path": "cases/nlp_kompas/test.csv", # Isi None jika tidak ada test set terpisah
    
    # Kolom: Set None untuk auto-detect, atau isi nama kolom secara eksplisit
    "text_cols": None, 
    "target_col": None,
    "id_col": None,
    
    # Preprocessing
    "lower": True,
    "strip_html": True,
    "strip_urls": True,
    "normalize_repeated_chars": True,
    "prefix_tags": True,
    
    # Modeling
    "max_features": 25000,
    "n_splits": 5,
    "reg_c": 1.0
}

print("Konfigurasi tersimpan.")


## 2. Auto-Discovery & Data Ingestion Engine


In [ ]:
def load_multiclass_data(cfg):
    tr_p, te_p = cfg["train_path"], cfg["test_path"]
    if not os.path.exists(tr_p):
        print(f"Peringatan: Berkas '{tr_p}' belum ada. Menjalankan fallback generator multi-kelas...")
        train_df = pd.DataFrame({
            "doc_id": range(1, 16),
            "text_content": [
                "pemain timnas sepak bola mencetak gol di babak kedua piala asia",
                "pertandingan bola liga inggris berlangsung sengit tadi malam",
                "ihsg ditutup menguat didorong saham perbankan dan emiten tambang",
                "bank indonesia mempertahankan suku bunga acuan untuk stabilkan inflasi",
                "smartphone terbaru ini dibekali chipset kecerdasan buatan bertenaga ai",
                "perusahaan teknologi merilis sistem operasi baru untuk laptop",
                "resep membuat masakan ayam rendang khas minang yang empuk dan lezat",
                "kuliner kopi lokal kini makin digemari kalangan muda di berbagai kafe",
                "peneliti lapan mengamati fenomena hujan meteor di langit malam",
                "teleskop luar angkasa menemukan planet baru berpotensi layak huni",
                "vaksinasi gratis dibuka untuk masyarakat umum di puskesmas",
                "tips menjaga kesehatan jantung dengan rajin berolahraga kardio",
                "wisata pantai di pesisir selatan menyajikan pemandangan sunset indah",
                "penerbangan internasional ke destinasi wisata bali mulai meningkat",
                "kementerian pendidikan meluncurkan beasiswa kuliah untuk mahasiswa berprestasi"
            ],
            "rubrik_kategori": [
                "Bola", "Bola", "Money", "Money", "Tekno", "Tekno", "Food", "Food",
                "Sains", "Sains", "Health", "Health", "Travel", "Travel", "Edukasi"
            ]
        })
        test_df = None
    else:
        ext = os.path.splitext(tr_p)[1].lower()
        if ext in ['.xlsx', '.xls']: train_df = pd.read_excel(tr_p); test_df = pd.read_excel(te_p) if te_p and os.path.exists(te_p) else None
        elif ext == '.parquet': train_df = pd.read_parquet(tr_p); test_df = pd.read_parquet(te_p) if te_p and os.path.exists(te_p) else None
        else: train_df = pd.read_csv(tr_p); test_df = pd.read_csv(te_p) if te_p and os.path.exists(te_p) else None

    # Auto ID
    id_col = cfg["id_col"]
    if not id_col or id_col not in train_df.columns:
        id_cands = [c for c in train_df.columns if re.search(r"id|uuid|kode|no|link", c, re.IGNORECASE)]
        id_col = id_cands[0] if id_cands else train_df.columns[0]

    # Auto Target
    target_col = cfg["target_col"]
    if not target_col or target_col not in train_df.columns:
        target_cands = [c for c in train_df.columns if c != id_col and re.search(r"target|label|kategori|category|biro|class|rubrik", c, re.IGNORECASE)]
        if not target_cands:
            target_cands = [c for c in train_df.columns if c != id_col and train_df[c].nunique() < 50]
        target_col = target_cands[0] if target_cands else train_df.columns[-1]

    # Auto Text
    text_cols = cfg["text_cols"]
    if not text_cols:
        text_cols = [c for c in train_df.columns if c not in [id_col, target_col] and (train_df[c].dtype == object or pd.api.types.is_string_dtype(train_df[c]))]
        if not text_cols: text_cols = [train_df.columns[1]]
    elif isinstance(text_cols, str):
        text_cols = [text_cols]

    return train_df, test_df, id_col, target_col, text_cols

train_df, test_df, ID_COL, TARGET_COL, TEXT_COLS = load_multiclass_data(CONFIG)
train_df = train_df.dropna(subset=[TARGET_COL] + TEXT_COLS).reset_index(drop=True)

categories = sorted(train_df[TARGET_COL].unique())
num_classes = len(categories)
label2id = {c: i for i, c in enumerate(categories)}
id2label = {i: c for i, c in enumerate(categories)}
train_df["label"] = train_df[TARGET_COL].map(label2id)

print(f"=== SKEMA MULTI-CLASS BERHASIL DIMUAT ===")
print(f"ID Col      : '{ID_COL}'")
print(f"Target Col  : '{TARGET_COL}' (Total {num_classes} Kelas)")
print(f"Text Col(s) : {TEXT_COLS}")
print(f"Total Baris : {len(train_df)}")
print(f"\nDistribusi 5 Kelas Teratas:\n{train_df[TARGET_COL].value_counts().head(5)}")


## 3. Modular Text Cleaning & Field Fusion


In [ ]:
def clean_str(s):
    if not isinstance(s, str): return ""
    if CONFIG["strip_html"]: s = re.sub(r"<[^>]+>", " ", s)
    if CONFIG["strip_urls"]: s = re.sub(r"https?://\S+|www\.\S+", " ", s)
    if CONFIG["normalize_repeated_chars"]: s = re.sub(r"(.)\1{2,}", r"\1", s)
    if CONFIG["lower"]: s = s.lower()
    return re.sub(r"\s+", " ", s).strip()

def fuse_columns(df, cols, prefix_tags=True):
    series_list = []
    for c in cols:
        cleaned = df[c].fillna("").astype(str).apply(clean_str)
        if prefix_tags and len(cols) > 1:
            tag = re.sub(r"[^\w]", "", c).upper()
            cleaned = f"[{tag}] " + cleaned
        series_list.append(cleaned)
    fused = series_list[0]
    for s in series_list[1:]: fused = fused + " " + s
    return fused.str.strip()

train_df["clean_text"] = fuse_columns(train_df, TEXT_COLS, CONFIG["prefix_tags"])
if test_df is not None:
    test_df["clean_text"] = fuse_columns(test_df, TEXT_COLS, CONFIG["prefix_tags"])

print("Contoh Teks Hasil Fusion:")
print(train_df["clean_text"].iloc[0][:150])


## 4. Rare Class Shielding & Pembagian Data Latih-Validasi


In [ ]:
# Proteksi terhadap kelas yang beranggotakan hanya 1 sampel
class_counts = train_df["label"].value_counts()
rare_classes = class_counts[class_counts < 2].index.tolist()

if rare_classes:
    rare_names = [id2label[rc] for rc in rare_classes]
    print(f"Rare Class Shielding aktif. Mengamankan kelas langka: {rare_names}")
    rare_df = train_df[train_df["label"].isin(rare_classes)]
    strat_df = train_df[~train_df["label"].isin(rare_classes)].reset_index(drop=True)
    
    tr_sub, val_df = train_test_split(strat_df, test_size=0.2, random_state=SEED, stratify=strat_df["label"])
    tr_df = pd.concat([tr_sub, rare_df], axis=0).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
else:
    tr_df, val_df = train_test_split(train_df, test_size=0.2, random_state=SEED, stratify=train_df["label"])

tr_df = tr_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(f"Data Latih: {len(tr_df)} baris | Data Validasi: {len(val_df)} baris")


## 5. Model 1: Sublinear TF-IDF + Balanced Logistic Regression (Baseline Cepat)


In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=CONFIG["max_features"], sublinear_tf=True)
X_tr = tfidf.fit_transform(tr_df["clean_text"])
X_va = tfidf.transform(val_df["clean_text"])

clf_baseline = LogisticRegression(
    C=CONFIG["reg_c"],
    max_iter=500,
    class_weight="balanced", # Memastikan kelas minoritas terlindungi
    random_state=SEED
)
clf_baseline.fit(X_tr, tr_df["label"])

baseline_preds = clf_baseline.predict(X_va)
acc = accuracy_score(val_df["label"], baseline_preds)
mf1 = f1_score(val_df["label"], baseline_preds, average="macro", zero_division=0)

print(f"=== PERFORMA BASELINE MODEL ===")
print(f"Akurasi Validasi : {acc * 100:.2f}%")
print(f"Macro-F1 Score   : {mf1 * 100:.2f}%")


## 6. Model 2: IndoBERT / RoBERTa dengan Adaptive Weighted Loss (GPU RTX 3070)


In [ ]:
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# 1. Hitung Balanced Class Weights dinamis
raw_weights = compute_class_weight(class_weight="balanced", classes=np.arange(num_classes), y=tr_df["label"].values)
clamped_weights = np.clip(raw_weights, a_min=0.2, a_max=30.0) # Proteksi gradien ekstrem
weights_tensor = torch.tensor(clamped_weights, dtype=torch.float).to(device)

MODEL_DIR = "./models/indobert-base-p1"
if not os.path.exists(MODEL_DIR):
    MODEL_DIR = "indobenchmark/indobert-base-p1"

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

    class PyTorchTextDataset(Dataset):
        def __init__(self, texts, labels, tokenizer, max_len=256):
            self.texts = list(texts)
            self.labels = list(labels)
            self.tokenizer = tokenizer
            self.max_len = max_len
        def __len__(self): return len(self.texts)
        def __getitem__(self, idx):
            enc = self.tokenizer(str(self.texts[idx]), truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt")
            return {
                "input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "labels": torch.tensor(self.labels[idx], dtype=torch.long)
            }

    train_dataset = PyTorchTextDataset(tr_df["clean_text"], tr_df["label"], tokenizer, max_len=256)
    val_dataset = PyTorchTextDataset(val_df["clean_text"], val_df["label"], tokenizer, max_len=256)

    transformer_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_DIR, num_labels=num_classes, id2label=id2label, label2id=label2id
    )

    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            loss_fct = torch.nn.CrossEntropyLoss(weight=weights_tensor)
            loss = loss_fct(outputs.get("logits").view(-1, self.model.config.num_labels), labels.view(-1))
            return (loss, outputs) if return_outputs else loss

    def compute_metrics(eval_pred):
        preds = np.argmax(eval_pred[0], axis=-1)
        return {
            "accuracy": accuracy_score(eval_pred[1], preds),
            "f1_macro": f1_score(eval_pred[1], preds, average="macro", zero_division=0)
        }

    training_args = TrainingArguments(
        output_dir="./checkpoints_multiclass_universal",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        weight_decay=0.01,
        fp16=torch.cuda.is_available(),
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        logging_steps=50,
        save_total_limit=1
    )

    trainer = WeightedTrainer(
        model=transformer_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )
    print("Trainer Transformer siap. Jalankan 'trainer.train()' untuk memulai fine-tuning.")
except Exception as e:
    print("Catatan inisialisasi Transformer:", e)


## 7. Laporan Performa Detail Kategori & Pembuatan Submission


In [ ]:
print("=== LAPORAN EVALUASI PER KATEGORI (BASELINE) ===")
val_present_labels = sorted(val_df["label"].unique())
print(classification_report(
    val_df["label"],
    baseline_preds,
    labels=val_present_labels,
    target_names=[id2label[i] for i in val_present_labels],
    zero_division=0
))

# Generate Submission jika Test Set tersedia
if test_df is not None:
    X_te = tfidf.transform(test_df["clean_text"])
    test_preds_cls = clf_baseline.predict(X_te)
    
    sub = pd.DataFrame({
        ID_COL: test_df[ID_COL],
        TARGET_COL: [id2label[p] for p in test_preds_cls]
    })
    sub.to_csv("submission_multiclass_universal.csv", index=False)
    print("Berkas submission_multiclass_universal.csv berhasil disimpan.")
    print(sub.head())
